# FX.fact_ohlc — Simple CRUD Test

Insert, read, update, and delete a row from the existing `FX.fact_ohlc` table.

In [ ]:
from datetime import datetime, timezone
from decimal import Decimal

from sqlalchemy import text

from imdr.config.settings import get_settings
from imdr.connectors.mssql import MSSQLConnector

connector = MSSQLConnector(get_settings())
print("Connected:", connector.engine.url)

## 1. CREATE — Insert a row

In [ ]:
insert_sql = text("""
    INSERT INTO FX.fact_ohlc
        (ts, symbol, series, tenor, deal_type, pair_used,
         open_px, high_px, low_px, close_px, mid_px,
         mid_mean_px, mid_median_px, bid, ask, n_ticks)
    OUTPUT INSERTED.id
    VALUES
        (:ts, :symbol, :series, :tenor, :deal_type, :pair_used,
         :open_px, :high_px, :low_px, :close_px, :mid_px,
         :mid_mean_px, :mid_median_px, :bid, :ask, :n_ticks)
""")

params = dict(
    ts=datetime.now(timezone.utc),
    symbol="EURUSD",
    series="spot",
    tenor="ON",
    deal_type="outright",
    pair_used="EURUSD",
    open_px=Decimal("1.08400000"),
    high_px=Decimal("1.08600000"),
    low_px=Decimal("1.08200000"),
    close_px=Decimal("1.08450000"),
    mid_px=Decimal("1.08425000"),
    mid_mean_px=Decimal("1.08410000"),
    mid_median_px=Decimal("1.08420000"),
    bid=Decimal("1.08400000"),
    ask=Decimal("1.08450000"),
    n_ticks=150,
)

with connector.session() as session:
    result = session.execute(insert_sql, params)
    inserted_id = result.scalar()

print(f"Inserted row with id = {inserted_id}")

## 2. READ — Fetch the row back

In [ ]:
import pandas as pd

read_sql = text("SELECT * FROM FX.fact_ohlc WHERE id = :id")

with connector.session() as session:
    df = pd.read_sql(read_sql, session.connection(), params={"id": inserted_id})

df

## 3. UPDATE — Change close_px

In [ ]:
update_sql = text("""
    UPDATE FX.fact_ohlc
    SET close_px = :close_px
    WHERE id = :id
""")

with connector.session() as session:
    session.execute(update_sql, {"close_px": Decimal("1.09000000"), "id": inserted_id})

# Verify the update
with connector.session() as session:
    df = pd.read_sql(read_sql, session.connection(), params={"id": inserted_id})

print(f"Updated close_px = {df['close_px'].iloc[0]}")
df

## 4. DELETE — Remove the test row

In [ ]:
delete_sql = text("DELETE FROM FX.fact_ohlc WHERE id = :id")

with connector.session() as session:
    result = session.execute(delete_sql, {"id": inserted_id})

print(f"Deleted {result.rowcount} row(s)")

# Verify it's gone
with connector.session() as session:
    df = pd.read_sql(read_sql, session.connection(), params={"id": inserted_id})

print(f"Rows remaining with id={inserted_id}: {len(df)}")

In [ ]:
connector.dispose()
print("Done — connection pool closed.")